# 01 DATA PREP

In [4]:
# SENTINEL-1 DATE CHECK
import ee
import pandas as pd

ee.Authenticate()
ee.Initialize(project="uhisea")

# Rough box around the Klang Valley (west, south, east, north in degrees)
aoi = ee.Geometry.Rectangle([101.25, 2.60, 101.98, 3.45])

s1 = (ee.ImageCollection("COPERNICUS/S1_GRD")
      .filterBounds(aoi)
      .filterDate("2021-11-01", "2022-01-01")
      .filter(ee.Filter.eq("instrumentMode", "IW")))

def describe(img):
    cover = (img.geometry().intersection(aoi, ee.ErrorMargin(1000)).area(1000)
             .divide(aoi.area(1000)).multiply(100).round())
    return ee.Feature(None, {
        "time_myt": img.date().format("YYYY-MM-dd HH:mm", "Asia/Kuala_Lumpur"),
        "satellite": ee.String("S1").cat(ee.String(img.get("platform_number"))),
        "pass": img.get("orbitProperties_pass"),
        "orbit": img.get("relativeOrbitNumber_start"),
        "polarisation": ee.List(img.get("transmitterReceiverPolarisation")).join("+"),
        "aoi_cover_pct": cover,
    })

rows = [f["properties"] for f in s1.map(describe).getInfo()["features"]]
df = pd.DataFrame(rows).sort_values("time_myt")
print(len(df), "images")
print(df.to_string(index=False))

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


34 images
 aoi_cover_pct  orbit       pass polarisation satellite         time_myt
            76     70  ASCENDING        VV+VH       S1A 2021-11-02 19:34
            14     70  ASCENDING        VV+VH       S1A 2021-11-02 19:34
             6    164 DESCENDING        VV+VH       S1A 2021-11-09 07:04
             1    164 DESCENDING        VV+VH       S1A 2021-11-09 07:04
            29    172  ASCENDING        VV+VH       S1A 2021-11-09 19:26
            18    172  ASCENDING        VV+VH       S1A 2021-11-09 19:26
            76     70  ASCENDING        VV+VH       S1A 2021-11-14 19:34
            14     70  ASCENDING        VV+VH       S1A 2021-11-14 19:34
           100     91 DESCENDING        VV+VH       S1A 2021-11-16 06:56
             6    164 DESCENDING        VV+VH       S1A 2021-11-21 07:04
             1    164 DESCENDING        VV+VH       S1A 2021-11-21 07:04
            29    172  ASCENDING        VV+VH       S1A 2021-11-21 19:26
            18    172  ASCENDING        V

In [5]:
# QUICK CHECKS ON OTHER SATELLITE IMAGES
collections = {
    "Sentinel-2":   ("COPERNICUS/S2_SR_HARMONIZED", "CLOUDY_PIXEL_PERCENTAGE"),
    "Landsat 8":    ("LANDSAT/LC08/C02/T1_L2", "CLOUD_COVER"),
    "Landsat 9":    ("LANDSAT/LC09/C02/T1_L2", "CLOUD_COVER"),
    "ALOS-2 ScanSAR": ("JAXA/ALOS/PALSAR-2/Level2_2/ScanSAR", None),
}

for name, (col_id, cloud_prop) in collections.items():
    col = ee.ImageCollection(col_id).filterBounds(aoi).filterDate("2021-12-16", "2021-12-31")
    def describe(img, cloud_prop=cloud_prop):
        props = {"time_myt": img.date().format("YYYY-MM-dd HH:mm", "Asia/Kuala_Lumpur")}
        if cloud_prop:
            props["cloud_pct"] = img.get(cloud_prop)
        return ee.Feature(None, props)
    rows = [f["properties"] for f in col.map(describe).getInfo()["features"]]
    print(f"\n{name}: {len(rows)} images")
    if rows:
        print(pd.DataFrame(rows).sort_values("time_myt").to_string(index=False))


Sentinel-2: 12 images
 cloud_pct         time_myt
 99.771268 2021-12-19 11:47
 99.248525 2021-12-19 11:47
 99.944578 2021-12-19 11:47
100.000000 2021-12-19 11:47
 78.220014 2021-12-24 11:47
 50.416906 2021-12-24 11:47
 92.385902 2021-12-24 11:47
 63.696207 2021-12-24 11:47
 91.975865 2021-12-29 11:47
 91.094692 2021-12-29 11:47
 76.333279 2021-12-29 11:47
 91.053783 2021-12-29 11:47

Landsat 8: 2 images
 cloud_pct         time_myt
     38.57 2021-12-24 11:28
     53.92 2021-12-24 11:28

Landsat 9: 2 images
 cloud_pct         time_myt
     96.92 2021-12-16 11:28
     62.20 2021-12-25 11:22

ALOS-2 ScanSAR: 4 images
        time_myt
2021-12-24 12:49
2021-12-24 12:50
2021-12-29 12:56
2021-12-29 12:57


In [ ]:
# CHECKS ON THE ENVIRONMENT LIBRARIES INSTALLATION
import pandas as pd, geopandas as gpd, pydeck as pdk, h3, osmnx as ox

print("pandas", pd.__version__)
print("geopandas", gpd.__version__)
print("pydeck", pdk.__version__)
print("h3", h3.__version__)
print("osmnx", ox.__version__)

# H3 test: the resolution 8 hexagon containing KL Sentral
cell = h3.latlng_to_cell(3.1337, 101.6869, 8)
print("KL Sentral hexagon:", cell, "| neighbours:", len(h3.grid_disk(cell, 1)) - 1)

pandas 3.0.6
geopandas 1.2.0
pydeck 0.9.3
h3 4.5.0
osmnx 2.1.1
KL Sentral hexagon: 8865050939fffff | neighbours: 6


___
**END**